[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/deployment.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239303-lesson-8-deployment)

# 部署（Deployment）

## 回顧

我們一路打造出一個具備記憶能力的 agent：

* `act`（行動）—— 讓模型呼叫特定的工具
* `observe`（觀察）—— 把工具的輸出回傳給模型
* `reason`（推理）—— 讓模型針對工具的輸出進行推理，決定接下來該怎麼做（例如再呼叫另一個工具，或是直接回覆）
* `persist state`（保存狀態）—— 使用記憶體中的 checkpointer，支援可被中斷的長時間對話

## 目標

接下來，我們會說明如何實際把 agent 部署到本機的 Studio，以及部署到 `LangGraph Cloud`。

In [3]:
%%capture --no-stderr
%pip install --quiet -U langgraph_sdk langchain_core

## 核心概念

這裡有幾個核心概念需要先理解 ——

`LangGraph` ——
- Python 與 JavaScript 函式庫
- 讓你能夠建立 agent 工作流程

`LangGraph API` ——
- 把 graph 的程式碼打包起來
- 提供一個 task queue 來管理非同步操作
- 提供 persistence（持久化），讓 state 能跨多次互動維持下來

`LangSmith Deployment`（前身為 `LangGraph Cloud`）——
- LangGraph API 的託管服務
- 讓你能從 GitHub repository 部署 graph
- 同時也為已部署的 graph 提供監控與追蹤
- 每個 deployment 都可透過一個專屬的 URL 存取

`LangSmith Studio`（前身為 `LangGraph Studio`）——
- 專為 LangGraph 應用打造的整合式開發環境（IDE）
- 以 API 作為後端，讓你能即時測試與探索 graph
- 可在本機執行，也可搭配雲端部署使用。詳見下文。

`LangGraph SDK` ——
- 用來以程式化方式與 LangGraph 的 graph 互動的 Python 函式庫
- 不論 graph 是在本機提供服務還是部署在雲端，都提供一致的操作介面
- 讓你能建立 client、存取 assistant、管理 thread，以及執行 run

## 在本機測試

## Studio

**⚠️ 注意**

在拍攝這些影片之後，我們更新了 Studio，現在它可以在本機執行並透過瀏覽器存取。相較於影片中所示的桌面版 App，這才是執行 Studio 比較建議的方式。它現在改名為 _LangSmith Studio_，不再叫做 _LangGraph Studio_。詳細的設定步驟可參考課程一開始的「Getting Setup」指南。你可以在[這裡](https://docs.langchain.com/langsmith/studio)看到 Studio 的說明，並在[這裡](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)找到本機部署的細節。
若要啟動本機的開發伺服器，請在本模組的 `/studio` 目錄下，於終端機執行下列指令：

```
langgraph dev
```

你應該會看到以下輸出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打開瀏覽器，前往上方顯示的 **Studio UI** URL。

In [1]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")

In [2]:
from langgraph_sdk import get_client

In [3]:
# 這是本機開發伺服器的 URL
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# 搜尋所有託管的 graph
assistants = await client.assistants.search()

In [4]:
assistants[-3]

{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
 'graph_id': 'agent',
 'config': {},
 'metadata': {'created_by': 'system'},
 'name': 'agent',
 'created_at': '2025-03-04T22:57:28.424565+00:00',
 'updated_at': '2025-03-04T22:57:28.424565+00:00',
 'version': 1}

In [3]:
# 我們建立一個 thread 來追蹤這次 run 的 state
thread = await client.threads.create()

接下來，我們可以[用 `client.runs.stream`](https://docs.langchain.com/oss/python/langgraph/graph-api/#stream-and-astream) 來執行我們的 agent，需要提供：

* `thread_id`
* `graph_id`
* `input`
* `stream_mode`

我們會在之後的模組深入討論 streaming。

現在，你只要先了解：我們是用 `stream_mode="values"` 來[串流（streaming）](https://docs.langchain.com/langsmith/streaming)取得 graph 在每個步驟後 state 的完整值。

state 會被記錄在 `chunk.data` 裡。

In [4]:
from langchain_core.messages import HumanMessage

# 輸入
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# 串流
async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {'example': False, 'additional_kwargs': {}, 'response_metadata': {}}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': 'cdbd7bd8-c476-4ad4-8ab7-4ad9e3654267', 'example': False}
{'content': '', 'additional_kwargs': {'tool_calls': [{'index': 0, 'id': 'call_iIPryzZZxRtXozwwhVtFObNO', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'multiply'}, 'type': 'function'}]}, 'response_metadata': {'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_157b3831f5'}, 'type': 'ai', 'name': None, 'id': 'run-06c7243c-426d-4c81-a113-f1335dda5fb2', 'example': False, 'tool_calls': [{'name': 'multiply', 'args': {'a': 3, 'b': 2}, 'id': 'call_iIPryzZZxRtXozwwhVtFObNO', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': None}
{'content': '6', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'tool', 'name': 'multiply', 'id': '988cb170-f6e6-43c1-82fd-309f519abe6d', 'tool_call_id': 'c

## 在 Cloud 上測試

我們可以透過 LangSmith 部署到 Cloud，做法如[這裡](https://docs.langchain.com/langsmith/deployment-quickstart#deploy-from-github-with-langgraph-cloud)所述。

### 在 GitHub 上建立一個新的 Repository

* 前往你的 GitHub 帳號
* 點擊右上角的「+」圖示，選擇 `"New repository"`
* 為你的 repository 命名（例如 `langchain-academy`）

### 把你的 GitHub Repository 加為遠端（Remote）

* 回到你在課程一開始 clone `langchain-academy` 的那個終端機
* 把你剛建立的 GitHub repository 加為遠端

```
git remote add origin https://github.com/your-username/your-repo-name.git
```
* 推送上去
```
git push -u origin main
```

### 把 LangSmith 連結到你的 GitHub Repository

* 前往 [LangSmith](hhttps://smith.langchain.com/)
* 點擊 LangSmith 左側面板的 `deployments` 分頁
* 點擊 `+ New Deployment`
* 接著，選擇你剛為課程建立的 Github repository（例如 `langchain-academy`）
* 把 `LangGraph API config file` 指向其中一個 `studio` 目錄
* 例如，module-1 請使用：`module-1/studio/langgraph.json`
* 設定你的 API key（例如，你可以直接從 `module-1/studio/.env` 檔案複製過來）

![Screenshot 2024-09-03 at 11.35.12 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fd61c93d48e5d0f47_deployment2.png)

### 使用你的 Deployment

接著，我們可以透過幾種不同的方式與我們的 deployment 互動：

* 使用 SDK，做法和先前一樣。
* 使用 [LangGraph Studio](https://docs.langchain.com/langsmith/deployment-quickstart#3-test-your-application-in-studio)。

![Screenshot 2024-08-23 at 10.59.36 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fa159a09a51d601de_deployment3.png)

若要在這份 notebook 中使用 SDK，只要確認 `LANGSMITH_API_KEY` 已設定即可！

In [1]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("LANGSMITH_API_KEY")

In [ ]:
# 把這裡換成你已部署 graph 的 URL
URL = "https://langchain-academy-8011c561878d50b1883f7ed11b32d720.default.us.langgraph.app"
client = get_client(url=URL)

# 搜尋所有託管的 graph
assistants = await client.assistants.search()

In [37]:
# 選擇這個 agent
agent = assistants[0]

In [38]:
agent

{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
 'graph_id': 'agent',
 'created_at': '2024-08-23T17:58:02.722920+00:00',
 'updated_at': '2024-08-23T17:58:02.722920+00:00',
 'config': {},
 'metadata': {'created_by': 'system'}}

In [40]:
from langchain_core.messages import HumanMessage

# 我們建立一個 thread 來追蹤這次 run 的 state
thread = await client.threads.create()

# 輸入
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# 串流
async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {'example': False, 'additional_kwargs': {}, 'response_metadata': {}}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '8ea04559-f7d4-4c82-89d9-c60fb0502f21', 'example': False}
{'content': '', 'additional_kwargs': {'tool_calls': [{'index': 0, 'id': 'call_EQoolxFaaSVU8HrTnCmffLk7', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'multiply'}, 'type': 'function'}]}, 'response_metadata': {'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_3aa7262c27'}, 'type': 'ai', 'name': None, 'id': 'run-b0ea5ddd-e9ba-4242-bb8c-80eb52466c76', 'example': False, 'tool_calls': [{'name': 'multiply', 'args': {'a': 3, 'b': 2}, 'id': 'call_EQoolxFaaSVU8HrTnCmffLk7', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': None}
{'content': '6', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'tool', 'name': 'multiply', 'id': '1bf558e7-79ef-4f21-bb66-acafbd04677a', 'tool_call_id': 'c